# Notebook 02 — Klasifikasi: Prediksi Status Order

**Fase 2 · Minilab EduBI · Data Mining**

---

## Tujuan
Membangun model klasifikasi untuk memprediksi **status order** (`done` vs `cancelled`)
menggunakan **Random Forest Classifier** dan mencatat hasilnya ke **MLflow**.

Hasil model membantu manajemen mengidentifikasi pola transaksi yang berpotensi dibatalkan,
sehingga dapat dilakukan intervensi lebih awal.

| Label | Kelas | Jumlah |
|-------|-------|--------|
| 1     | done     | ~80 |
| 0     | cancelled | ~20 |

---
## 1. Setup & Koneksi

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import clickhouse_connect
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

CH_HOST  = os.getenv('CH_HOST', 'localhost')
CH_PORT  = int(os.getenv('CH_PORT', 8123))
CH_USER  = os.getenv('CH_USER', 'default')
CH_PASS  = os.getenv('CH_PASSWORD', '')
MLFLOW_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('02_classification_order_status')

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASS
)
print('Koneksi ClickHouse berhasil.')

---
## 2. Load Data dari Silver Layer


In [ ]:
query = """
SELECT
    order_id,
    customer_id,
    category,
    quantity,
    toFloat64(unit_price)   AS unit_price,
    toFloat64(total_price)  AS total_price,
    order_year,
    order_month,
    branch,
    revenue_category,
    status
FROM silver.silver_sales
WHERE status IN ('done', 'cancelled')
"""

df = client.query_df(query)
print(f'Total records: {len(df)}')
print('\nDistribusi target (status):')
print(df['status'].value_counts())
df.head()

---
## 3. Feature Engineering


In [ ]:
df_model = df.copy()

# Target encoding: done=1, cancelled=0
df_model['target'] = (df_model['status'] == 'done').astype(int)

# Categorical encoding fitur
le = LabelEncoder()
for col in ['category', 'branch', 'revenue_category']:
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))

FEATURES = [
    'quantity', 'unit_price', 'total_price',
    'order_year', 'order_month',
    'category_enc', 'branch_enc', 'revenue_category_enc'
]
TARGET = 'target'

X = df_model[FEATURES]
y = df_model[TARGET]

print('Fitur:', FEATURES)
print('Shape X:', X.shape)
print('\nDistribusi target:')
print(y.value_counts().rename({1: "done", 0: "cancelled"}))

---
## 4. Train/Test Split


In [ ]:
# Stratify untuk menjaga proporsi done/cancelled di train & test
min_class_count = y.value_counts().min()
stratify_param = y if min_class_count >= 2 else None
if stratify_param is None:
    print('⚠ Stratify dinonaktifkan: kelas minoritas < 2 sampel')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_param
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print('Distribusi y_train:', y_train.value_counts().to_dict())
print('Distribusi y_test :', y_test.value_counts().to_dict())

---
## 5. Training & Evaluasi dengan MLflow


In [ ]:
PARAMS = {
    'n_estimators':     100,
    'max_depth':        5,
    'min_samples_leaf': 2,
    'class_weight':     'balanced',
    'random_state':     42
}

mlflow.set_experiment('02_classification_order_status')

with mlflow.start_run(run_name='random_forest_v1'):
    model = RandomForestClassifier(**PARAMS)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)

    metrics = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall':    recall_score(y_test, y_pred, zero_division=0),
        'f1':        f1_score(y_test, y_pred, zero_division=0),
    }

    # ROC-AUC: hanya jika kedua kelas ada di test set
    if len(set(y_test)) == 2 and len(model.classes_) == 2:
        y_proba = model.predict_proba(X_test)[:, 1]
        metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
    else:
        metrics['roc_auc'] = float('nan')
        print('⚠ ROC-AUC dilewati: tidak semua kelas ada di test set')

    mlflow.log_params(PARAMS)
    mlflow.log_metrics({k: v for k, v in metrics.items() if v == v})
    mlflow.sklearn.log_model(model, 'random_forest_model')

    print('\nMetrik evaluasi:')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}' if v == v else f'  {k:<12}: N/A')

print('\n✅ Run MLflow selesai')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Cancelled', 'Done']))

---
## 6. Confusion Matrix & Feature Importance


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Cancelled', 'Done'],
    cmap='Blues', ax=ax1
)
ax1.set_title('Confusion Matrix — Status Order')

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
importances.plot.barh(ax=ax2, color='steelblue')
ax2.set_title('Feature Importance')
ax2.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('experiments/classification_results.png', dpi=100)
plt.show()

---
## 7. Kesimpulan

**Pertanyaan Diskusi:**
1. Fitur apa yang paling berpengaruh dalam memprediksi status order?
2. Mengapa `class_weight='balanced'` penting untuk data 80:20?
3. Apa arti *precision* dan *recall* untuk kelas `cancelled` dalam konteks bisnis ini?
4. Jika model sering salah prediksi `cancelled` sebagai `done`, apa dampak bisnisnya?
5. Bagaimana cara meningkatkan performa jika data bertambah?

**Lihat hasil eksperimen di MLflow:** http://localhost:5000